# The same numbers

This notebook drives the scenario the command line drives, at the same seed, and compares
the two answers. The point is not that the numbers are interesting — they are the smoke
scenario's — but that there is one set of them. A binding that agreed with the terminal
only sometimes would leave no way to tell which surface produced a published figure.

Build and install the wheel first:

```bash
make sdk
```

In [ ]:
import json, os, subprocess
from pathlib import Path

import karman

# A scenario path and a report path are relative to the repository, so the notebook stands
# where the terminal stands rather than resolving them from wherever it was opened.
ROOT = Path.cwd().resolve()
while not (ROOT / "crates").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
SCENARIO = "testdata/scenarios/cpamm-smoke.json"

## What the notebook gets

In [ ]:
from_python = json.loads(karman.run(SCENARIO, runs=100, seed=42, report_dir="out/sdk"))
from_python

## What the terminal gets

In [ ]:
binary = ROOT / "target" / "release" / "karman"
if not binary.exists():
    binary = binary.with_suffix(".exe")
done = subprocess.run(
    [str(binary), "run", "--scenario", SCENARIO, "--runs", "100", "--seed", "42",
     "--report-dir", "out/cli"],
    cwd=ROOT, capture_output=True, text=True, check=True,
)
from_terminal = json.loads(done.stdout.strip().splitlines()[-1])
from_terminal

## They agree

Only the report path differs, because each surface was asked to write somewhere different.
Every measured number is the same, and it is the same because both calls went through one
composition rather than two that happen to agree today.

In [ ]:
from_python.pop("report"), from_terminal.pop("report")
assert from_python == from_terminal
from_python

## Varying a parameter

A scenario is the unit of description, and a sweep turns one of the knobs the schema knows
how to turn. Every point runs at the same seed, so two points differ by the knob and by
nothing else.

In [ ]:
json.loads(karman.sweep(
    SCENARIO, vary="group-count:0", values=[1, 5, 20], runs=50, seed=42,
    report_dir="out/sdk",
))

## Asking what may be claimed

The ladder reports on all four of its levels, including the ones this mechanism cannot
reach. A report listing only what was measured would read as though the rest had passed.

In [ ]:
json.loads(karman.verify("testdata/scenarios/cpamm-mixed.json", seed=42, report_dir="out/sdk"))